# RFdiffusion3 Structure Design

![RFdiffusion3 Structure Design](https://proto-bio.github.io/proto-assets/images/tool/rfdiffusion3/hero.png)

This notebook demonstrates de novo protein structure design using RFdiffusion3. We generate a novel protein backbone from scratch by specifying a target length, then inspect the designed sequence and structure returned by the model.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("rfdiffusion3")
display_overview("rfdiffusion3")
display_docs_section("rfdiffusion3", "Background")

# RFdiffusion3

Released in 2025 by the [Baker Lab at the Institute for Protein Design](https://www.ipd.uw.edu/), RFdiffusion3 is an all-atom denoising-diffusion generative model for de novo protein design. Given constraints such as a target binding site, a structural motif, a small-molecule pocket, or a symmetry group, it generates novel protein structures together with compatible amino-acid sequences. It is the all-atom successor to RFdiffusion and addresses protein binders, nucleic-acid and small-molecule binders, enzymes, and symmetric assemblies within a single model.

RFdiffusion3 ([Butcher et al., 2025](https://doi.org/10.1101/2025.09.18.676967)) is a denoising-diffusion generative model trained to design protein structures at all-atom resolution under arbitrary spatial constraints. Starting from random noise, it iteratively denoises atomic coordinates toward a plausible protein while jointly refining the underlying amino-acid sequence. Training combines structures from the [Protein Data Bank](https://www.rcsb.org/) with multi-task conditioning, in which each training example is presented with a randomly generated design problem that constrains a sampled combination of motif tokens, atom subsets, residue identities, or sequence-index labels. The model is therefore trained jointly on binder design, motif scaffolding, inverse folding, sidechain placement, and prediction-style tasks under a single objective, and a single trained checkpoint supports every conditioning style at inference.

RFdiffusion3 is the successor to [RFdiffusion](https://doi.org/10.1038/s41586-023-06415-8) (Watson et al., 2023), which diffused only over the backbone N, Cα, C, and O atoms and required [ProteinMPNN](https://doi.org/10.1126/science.add2187) as a separate sequence-design step. By denoising every atom and co-designing the sequence, RFdiffusion3 incorporates small-molecule pockets, hydrogen-bond donor and acceptor patterning, and explicit nucleotide and ligand context directly into the generative process. It is the structure-design model within the [Foundry](https://github.com/RosettaCommons/foundry) framework, which distributes it alongside [RoseTTAFold3](https://doi.org/10.1101/2025.08.14.670328) for structure prediction and [ProteinMPNN](https://bio-pro.mintlify.app/tools/inverse-folding/proteinmpnn) for inverse folding.

## Available tools

In [2]:
display_available_tools("rfdiffusion3")

- **`run_rfdiffusion3()`** — De novo protein structure design using RFdiffusion3

### `run_rfdiffusion3`

RFdiffusion3 is an all-atom diffusion model that co-designs protein backbone structure and amino acid sequence. Starting from a design specification — which can be as simple as a target length or as detailed as a set of motif constraints and hotspot residues — the model iteratively denoises random coordinates into a physically plausible protein. This example uses the simplest case: unconditional generation of a 100-residue protein with no conditioning information, illustrating the full input-config-output cycle.

In [3]:
from proto_tools import (
    RFdiffusion3Config,
    RFdiffusion3DesignSpec,
    RFdiffusion3Input,
    run_rfdiffusion3,
)

In [4]:
# Display input docs
display_api_reference("rfdiffusion3", "input", "run_rfdiffusion3")

# Unconditional design: generate a 100-residue protein from scratch
inputs = RFdiffusion3Input(
    design_specs=[
        RFdiffusion3DesignSpec(length="100")
    ]
)

**Input** — `RFdiffusion3Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `design_specs` | `list[proto_tools.tools.structure_design.rfdiffusion3.rfdiffusion3_sample.RFdiffusion3DesignSpec]` | `[]` | List of design specifications |
| `raw_json` | `str \| None` | `None` | Raw JSON string for advanced RFdiffusion3 configuration |

In [5]:
# Display config docs
display_api_reference("rfdiffusion3", "config", "run_rfdiffusion3")

# Use a single sample with reduced timesteps to keep the example fast
config = RFdiffusion3Config(
    n_batches=1,
    diffusion_batch_size=1,
    num_timesteps=50,
    step_scale=1.5,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `RFdiffusion3Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `n_batches` | `int` | `1` | Independent batches per spec (total designs per spec = n_batches * diffusion_batch_size) |
| `diffusion_batch_size` | `int` | `8` | Designs sampled in parallel per batch |
| `num_timesteps` | `int` | `200` | Diffusion timesteps for sampling (more = slower, generally higher quality) |
| `step_scale` | `float` | `1.5` | Diffusion step size scaling; higher = less diverse, more designable (typical: 1.0-2.0) |
| `sampler_kind` | `Literal['default', 'symmetry']` | `'default'` | Inference sampler kind ('symmetry' for homo-oligomer design) |
| `center_option` | `Literal['all', 'motif', 'diffuse']` | `'all'` | Coordinate-frame centering mode (all/motif/diffuse) |
| `use_classifier_free_guidance` | `bool` | `False` | Enable CFG sampling (cfg_scale is a no-op when False) |
| `cfg_scale` | `float` | `1.5` | CFG scale (typical: 1.0-3.0); requires use_classifier_free_guidance=True |
| `cfg_features` | `list[Literal['active_donor', 'active_acceptor', 'ref_atomwise_rasa']] \| None` | `None` | CFG steering feature names; None uses upstream default (donor/acceptor/RASA) |
| `cfg_t_max` | `float \| None` | `None` | Maximum diffusion timestep (0.0-1.0) at which CFG is applied |
| `gamma_0` | `float` | `0.6` | Sampler stochasticity (lower = more designable, less diverse); >0.5 for symmetry sampler |
| `low_memory_mode` | `bool` | `False` | Memory-efficient tokenization (slower); set True only if GPU RAM is tight |
| `dump_trajectories` | `bool` | `False` | Save diffusion trajectory frames to the output directory |
| `align_trajectory_structures` | `bool` | `False` | Align trajectory structures across timesteps |
| `prevalidate_inputs` | `bool` | `False` | Validate the full input JSON before launching diffusion |
| `ckpt_path` | `str` | `'rfd3'` | Checkpoint path or canonical alias (default: rfd3 production preset) |
| `input_dir` | `str \| None` | `None` | Optional input directory for local execution inputs |
| `output_dir` | `str \| None` | `None` | Optional output directory for local execution outputs |

In [6]:
# Run the design
output = run_rfdiffusion3(inputs, config)
n_designs = sum(len(bundle) for bundle in output.designed_structures)
print(f"Generated {n_designs} designs across {len(output)} input specs")

Running run_rfdiffusion3 [00:00]

Generated 1 designs across 1 input specs


> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [7]:
# Display output docs and inspect results
display_api_reference("rfdiffusion3", "output", "run_rfdiffusion3")

# Output is grouped per input spec: each bundle holds the N designs for one spec.
bundle = output[0]
first = bundle[0]
print(f"Spec key: {bundle.spec_key}")
print(f"Designs in bundle: {len(bundle)}")
print(f"Chains: {len(first.complex.chains)}")
print(f"Sequence length: {len(first.complex.chains[0].sequence)}")
print(f"Sequence: {first.complex.chains[0].sequence}")
print(f"Metadata keys: {list(first.metadata.keys())}")

# Visualize the first designed structure of the first bundle
first.structure.visualize()

**Output** — `RFdiffusion3Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_structures` | `list[proto_tools.tools.structure_design.rfdiffusion3.rfdiffusion3_sample.RFdiffusion3Designs]` | `[]` | Per-spec bundles of designed structures |

Spec key: spec-0
Designs in bundle: 1
Chains: 1
Sequence length: 100
Sequence: AASITAASLAGTTLTVSVANSAGAATTLTVTVTNAAGAAVAAAIAGTNAASASLSTAAAASITLAVPAAATATVTLALPAAGSYTVTVTDAAGNTASLTV
Metadata keys: ['diffused_index_map', 'metrics', 'specification', 'ckpt_path', 'seed']


## Export Results

Designed structures can be exported to PDB or mmCIF format. PDB is widely compatible with visualization tools such as PyMOL and ChimeraX, while mmCIF supports richer metadata and is preferred for larger or multi-chain structures.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("rfdiffusion3_designs_pdb", export_path=output_dir, file_format="pdb")
print(f"PDB designs exported to {output_dir / 'rfdiffusion3_designs_pdb/'}")

output.export("rfdiffusion3_designs_cif", export_path=output_dir, file_format="cif")
print(f"CIF designs exported to {output_dir / 'rfdiffusion3_designs_cif/'}")

PDB designs exported to example_output/rfdiffusion3_designs_pdb
CIF designs exported to example_output/rfdiffusion3_designs_cif
